# 2강: SQLite FTS5 역색인 실습

이 노트북에서는 SQLite FTS5(Full-Text Search 5) 역색인의 원리를 배우고,
SeekSeek이 사용하는 external content 모드와 트리거 동기화를 직접 실습합니다.

## 2.1 역색인(Inverted Index)이란?

역색인은 **"단어 → 해당 단어를 포함하는 문서 목록"** 매핑입니다.

```
일반 인덱스 (정방향):  문서1 → [python, search, file]
역색인 (역방향):       python → [문서1, 문서3, 문서7]
                       search → [문서2, 문서3, 문서5]
```

역색인 덕분에 LIKE '%keyword%' 대신 O(log N) 이하로 검색할 수 있습니다.

## 2.2 기본 FTS5 테이블 만들기

In [ ]:
import sqlite3

# 메모리 DB로 실습
conn = sqlite3.connect(':memory:')

# FTS5 가상 테이블 생성
conn.execute('CREATE VIRTUAL TABLE docs USING fts5(title, body)')

# 샘플 데이터 삽입
sample_docs = [
    ('Python 입문', 'Python은 배우기 쉬운 프로그래밍 언어입니다.'),
    ('검색 엔진 구현', 'FTS5를 사용하여 전문 검색 엔진을 구현합니다.'),
    ('파일 관리', 'NTFS MFT를 파싱하여 파일 목록을 관리합니다.'),
    ('Python 검색', 'Python으로 역색인 기반 검색 시스템을 만듭니다.'),
    ('문서 추출', 'PDF와 DOCX에서 텍스트를 추출하는 방법을 설명합니다.'),
]

conn.executemany('INSERT INTO docs(title, body) VALUES (?, ?)', sample_docs)
print(f'{len(sample_docs)}개 문서 삽입 완료')

## 2.3 MATCH 쿼리 — 기본 검색

In [ ]:
# 기본 MATCH 검색
print('=== "Python" 검색 ===')
for row in conn.execute("SELECT rowid, title, body FROM docs WHERE docs MATCH 'Python'"):
    print(f'  [{row[0]}] {row[1]}: {row[2][:50]}...')

print()
print('=== "검색" 검색 ===')
for row in conn.execute("SELECT rowid, title, body FROM docs WHERE docs MATCH '검색'"):
    print(f'  [{row[0]}] {row[1]}: {row[2][:50]}...')

## 2.4 BM25 랭킹

BM25(Best Matching 25)는 TF-IDF의 확장으로, 문서 길이를 고려한 랭킹 알고리즘입니다.

$$\text{BM25}(D, Q) = \sum_{i} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}$$

- IDF = 역문서빈도 (흔하지 않은 단어에 높은 가중치)
- f(qi, D) = 문서 D에서 토큰 qi의 출현 횟수
- k1 = 1.2, b = 0.75 (기본값)
- FTS5의 `rank` 컬럼은 bm25() 반환값(음수)을 담고 있어 ORDER BY rank ASC = 관련성 높은 순

In [ ]:
# BM25 랭킹 사용
print('=== BM25 랭킹으로 "Python" 검색 ===')
sql = """
    SELECT rowid, rank, title, body
    FROM docs
    WHERE docs MATCH 'Python'
    ORDER BY rank  -- rank는 음수 → 작을수록 관련성 높음
"""
for row in conn.execute(sql):
    print(f'  rank={row[1]:.4f}  [{row[0]}] {row[2]}')

## 2.5 FTS5 확장 쿼리 문법

| 문법 | 설명 | 예시 |
|------|------|------|
| 단순 단어 | 해당 단어 포함 | `python` |
| `AND` | 두 조건 모두 충족 | `python AND 검색` |
| `OR` | 하나 이상 충족 | `PDF OR DOCX` |
| `NOT` | 제외 | `python NOT 입문` |
| `"..."` | 구문 검색 | `"역색인 기반"` |
| `token*` | 접두사 매칭 | `검색*` |
| `NEAR(a b, N)` | N 토큰 이내 근접 | `NEAR(python 검색, 5)` |
| `^토큰` | 첫 토큰만 매칭 | `^Python` |

In [ ]:
queries = [
    ("AND 검색", "Python AND 검색"),
    ("OR 검색", "PDF OR DOCX"),
    ("NOT 검색", "Python NOT 입문"),
    ("접두사 검색", '"검색"*'),
]

for label, query in queries:
    print(f'=== {label}: {query} ===')
    try:
        for row in conn.execute('SELECT rowid, title FROM docs WHERE docs MATCH ?', (query,)):
            print(f'  [{row[0]}] {row[1]}')
    except sqlite3.OperationalError as e:
        print(f'  오류: {e}')
    print()

## 2.6 External Content 모드 (SeekSeek 방식)

SeekSeek은 `content=files, content_rowid=id` 옵션으로 FTS5를 사용합니다.
이 방식은 원본 텍스트를 FTS5에 중복 저장하지 않아 공간을 절약합니다.
대신 INSERT/DELETE/UPDATE 트리거로 수동 동기화가 필요합니다.

In [ ]:
# External Content 모드 실습
conn2 = sqlite3.connect(':memory:')

# 원본 테이블
conn2.execute('CREATE TABLE files (id INTEGER PRIMARY KEY, name TEXT, path TEXT)')

# FTS5 외부 콘텐츠 테이블
conn2.execute("""
    CREATE VIRTUAL TABLE fts_files
    USING fts5(name, path, content=files, content_rowid=id)
""")

# 트리거 설정 (SeekSeek의 indexer.py와 동일한 방식)
conn2.executescript("""
    -- INSERT 시 FTS5에도 등록
    CREATE TRIGGER files_ai AFTER INSERT ON files BEGIN
        INSERT INTO fts_files(rowid, name, path)
        VALUES (new.id, new.name, new.path);
    END;

    -- DELETE 시 FTS5에서도 제거 (특수 DELETE 구문 사용)
    CREATE TRIGGER files_ad AFTER DELETE ON files BEGIN
        INSERT INTO fts_files(fts_files, rowid, name, path)
        VALUES ('delete', old.id, old.name, old.path);
    END;

    -- UPDATE 시 구값 제거 + 신값 등록 (2단계)
    CREATE TRIGGER files_au AFTER UPDATE ON files BEGIN
        INSERT INTO fts_files(fts_files, rowid, name, path)
        VALUES ('delete', old.id, old.name, old.path);
        INSERT INTO fts_files(rowid, name, path)
        VALUES (new.id, new.name, new.path);
    END;
""")

print('External content 테이블 + 트리거 생성 완료')

In [ ]:
# 데이터 삽입 → 트리거가 자동으로 FTS5에 등록
conn2.execute("INSERT INTO files VALUES (1, 'report.docx', 'C:\\Documents\\report.docx')")
conn2.execute("INSERT INTO files VALUES (2, 'main.py', 'C:\\Projects\\main.py')")
conn2.execute("INSERT INTO files VALUES (3, 'readme.md', 'C:\\Projects\\readme.md')")

# FTS5 검색
print('=== "main" 검색 ===')
for row in conn2.execute('SELECT rowid, * FROM fts_files WHERE fts_files MATCH ?', ('main',)):
    print(f'  rowid={row[0]}, name={row[1]}, path={row[2]}')

# 파일 삭제 → 트리거가 FTS5에서도 제거
conn2.execute('DELETE FROM files WHERE id = 2')

print('\n=== 삭제 후 "main" 검색 ===')
results = conn2.execute('SELECT rowid FROM fts_files WHERE fts_files MATCH ?', ('main',)).fetchall()
print(f'  결과 수: {len(results)}  (삭제 반영됨)')

## 2.7 SeekSeek의 쿼리 빌더 실습

SeekSeek의 `_build_fts_query()`는 일반 텍스트를 FTS5 prefix 쿼리로 변환합니다.

In [ ]:
import re

_FTS5_NATIVE = re.compile(r'\b(AND|OR|NOT)\b|NEAR\s*\(|"|\^', re.IGNORECASE)

def build_fts_query(query: str) -> str:
    """사용자 입력 → FTS5 쿼리 변환 (SeekSeek 방식)"""
    q = query.strip()
    if not q:
        return ''
    # 네이티브 FTS5 연산자가 있으면 그대로 반환
    if _FTS5_NATIVE.search(q):
        return q
    # 일반 텍스트 → prefix 검색
    parts = []
    for w in q.split():
        clean = ''.join(c for c in w.rstrip('*') if c.isalnum() or c in '._-가-힣')
        if clean:
            parts.append(f'"{ clean }"*')
    return ' '.join(parts)

# 테스트
test_queries = [
    'main.py',
    'my document',
    'A AND B',
    '"exact phrase"',
    'report*',
]

for q in test_queries:
    print(f'  "{q}" → "{build_fts_query(q)}"')

In [ ]:
# 정리
conn.close()
conn2.close()
print('DB 연결 종료')

## 요약

- **역색인**: "단어 → 문서ID 목록" 매핑으로 O(log N) 검색 가능
- **FTS5**: SQLite의 전문 검색 확장, MATCH 쿼리 + BM25 랭킹 지원
- **External Content**: `content=테이블명`으로 중복 저장 없이 FTS5 사용
- **트리거 동기화**: INSERT/DELETE/UPDATE 트리거로 원본 ↔ FTS5 동기화
- **BM25**: TF-IDF 확장, 문서 길이 정규화로 관련성 높은 순서 정렬